# ACE Mission Implementation: L1 Halo Orbit and Composition Analysis
## ACE 미션 구현: L1 Halo 궤도와 조성 분석

**Paper**: Stone et al. (1998), *The Advanced Composition Explorer*, Space Sci. Rev. 86, 1-22.

This notebook reproduces three central concepts of the ACE mission:

1. **L1 libration point geometry** — derive the Sun-Earth L1 distance and visualize the modified halo orbit ACE inhabits.
2. **Time-of-Flight (TOF) composition analysis** — simulate the ULEIS-style TOF vs energy plane and demonstrate isotope separation (e.g., $^{20}\!Ne$ vs $^{22}\!Ne$).
3. **dE/dx vs E composition analysis** — simulate the CRIS/SIS-style Bethe-Bloch curves separating elements by Z and isotopes by M.

이 노트북은 ACE 미션의 세 가지 핵심 개념을 재현한다:
1. **L1 라그랑주 점 기하** — Sun-Earth L1 거리 유도와 ACE의 modified halo 궤도 시각화.
2. **Time-of-Flight (TOF) 조성 분석** — ULEIS 스타일의 TOF vs 에너지 평면을 시뮬레이션하고 동위원소 분리 (예: $^{20}\!Ne$ vs $^{22}\!Ne$) 시연.
3. **dE/dx vs E 조성 분석** — CRIS/SIS 스타일의 Bethe-Bloch 곡선을 시뮬레이션하여 Z로 원소를, M으로 동위원소를 분리.

In [ ]:
# Standard scientific Python imports.
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import brentq

# Plot styling.
plt.rcParams["figure.dpi"] = 110
plt.rcParams["figure.figsize"] = (8.0, 5.0)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

---
## 1. L1 Libration Point Geometry / L1 라그랑주 점 기하

The collinear inner Lagrange point L1 lies between the Sun and Earth where centrifugal force in the co-rotating frame balances the net gravity. To leading order in the small mass ratio $\mu = m_E / (m_E + M_\odot) \approx 3 \times 10^{-6}$:

$$r_{L1} \approx R\,\left(\frac{\mu}{3}\right)^{1/3}$$

Sun-Earth 사이의 collinear L1 점은 회전 좌표계에서 원심력과 순중력이 균형을 이루는 곳에 위치한다. 소질량비 $\mu \approx 3 \times 10^{-6}$의 leading order 근사로 $r_{L1} \approx R (\mu/3)^{1/3}$.

We will solve the exact quintic balance equation numerically and compare with the leading-order approximation.

정확한 5차 균형 방정식을 수치적으로 풀고 leading-order 근사와 비교한다.

In [ ]:
def l1_balance_equation(r_over_R: float, mu: float) -> float:
    """Compute residual of L1 force-balance equation in the co-rotating frame.

    The collinear inner libration point L1 satisfies (for primary M1=Sun at the
    barycenter side, secondary M2=Earth):

        (1 - mu) / (1 - x)^2  -  mu / x^2  -  (1 - mu - x) = 0

    where x = r/R is the L1 distance from Earth in units of the Sun-Earth
    separation R, and mu = m_E / (m_E + M_sun).

    Args:
        r_over_R: Distance from Earth toward the Sun in units of R.
        mu: Reduced mass of the secondary (Earth).

    Returns:
        Residual of the balance equation (zero at L1).
    """
    x = r_over_R
    return (1.0 - mu) / (1.0 - x) ** 2 - mu / x ** 2 - (1.0 - mu - x)


# Sun-Earth system parameters.
M_SUN = 1.989e30          # kg
M_EARTH = 5.972e24        # kg
AU_KM = 1.495978707e8     # km
R_EARTH_KM = 6378.137     # km

MU = M_EARTH / (M_EARTH + M_SUN)
print(f"Mass ratio mu = m_E / (m_E + M_sun) = {MU:.6e}")

# Leading-order approximation.
r_l1_approx = (MU / 3.0) ** (1.0 / 3.0)
print(f"Leading-order  r_L1 / R = {r_l1_approx:.6e}")
print(f"               r_L1     = {r_l1_approx * AU_KM:.3e} km")
print(f"               r_L1     = {r_l1_approx * AU_KM / R_EARTH_KM:.1f} R_E")

# Exact root (Brent's method between sensible bounds).
x_exact = brentq(l1_balance_equation, 1e-5, 0.5, args=(MU,))
print(f"\nExact root     r_L1 / R = {x_exact:.6e}")
print(f"               r_L1     = {x_exact * AU_KM:.3e} km")
print(f"               r_L1     = {x_exact * AU_KM / R_EARTH_KM:.1f} R_E")

The exact value is ~1.5 × 10⁶ km ≈ 235 R_E, in agreement with the paper's quoted ~240 R_E (the paper rounds; some references use slightly different reduced masses).

정확값은 ~1.5 × 10⁶ km ≈ 235 R_E로, 논문이 인용한 ~240 R_E와 일치한다.

### 1.1 Solar wind transit time L1 → Earth / 태양풍의 L1 → 지구 전송 시간

In [ ]:
def sw_transit_time_minutes(distance_km: float, sw_speed_km_s: float) -> float:
    """Compute solar-wind transit time in minutes from L1 to Earth.

    Args:
        distance_km: L1 distance from Earth in kilometers.
        sw_speed_km_s: Solar-wind bulk speed in km/s.

    Returns:
        Transit time in minutes.
    """
    return distance_km / sw_speed_km_s / 60.0


r_l1_km = x_exact * AU_KM
for v_sw in [350, 400, 500, 700, 800]:
    t_min = sw_transit_time_minutes(r_l1_km, v_sw)
    print(f"v_SW = {v_sw:4d} km/s  ->  Δt = {t_min:5.1f} min")

Slow streams give ~60-70 min lead time; fast streams ~30-35 min. ACE delivers this directly to NOAA via the RTSW system at 1-min cadence.

느린 스트림은 ~60-70분, 빠른 스트림은 ~30-35분의 사전 경고 시간을 제공. ACE는 RTSW 시스템을 통해 1분 분해능으로 NOAA에 직접 전송.

### 1.2 Modified halo orbit visualization / Modified halo 궤도 시각화

The paper states the ACE halo orbit has major axis ~150,000 km and minor axis ~75,000 km. We model this as a Lissajous-like ellipse in the y-z plane perpendicular to the Sun-Earth line, with quasi-periodic motion.

논문에 따르면 ACE의 halo 궤도는 장축 ~150,000 km, 단축 ~75,000 km이다. 이를 태양-지구 선에 수직인 y-z 평면의 Lissajous-like 타원으로 모델링한다.

In [ ]:
def lissajous_halo_orbit(
    t_days: np.ndarray,
    a_km: float = 75000.0,
    b_km: float = 37500.0,
    period_y_days: float = 178.0,
    period_z_days: float = 178.0 * 1.05,
) -> tuple[np.ndarray, np.ndarray]:
    """Compute Lissajous-like ACE halo orbit in the y-z plane.

    Uses semi-axes (so full major = 2*a, full minor = 2*b). The ACE paper
    quotes a major axis of ~150,000 km and minor axis of ~75,000 km, which
    correspond to a = 75,000 km and b = 37,500 km here.

    Args:
        t_days: Array of times in days.
        a_km: Semi-major axis (y direction) in km.
        b_km: Semi-minor axis (z direction) in km.
        period_y_days: Orbital period in y direction (planar libration).
        period_z_days: Orbital period in z direction (vertical libration).

    Returns:
        Tuple (y_km, z_km) of orbit coordinates.
    """
    omega_y = 2.0 * np.pi / period_y_days
    omega_z = 2.0 * np.pi / period_z_days
    y_km = a_km * np.cos(omega_y * t_days)
    z_km = b_km * np.sin(omega_z * t_days)
    return y_km, z_km


# Span 5 years (1827 days) at 1-day resolution.
t_days = np.linspace(0.0, 1827.0, 5000)
y, z = lissajous_halo_orbit(t_days)

fig, ax = plt.subplots(figsize=(7, 6))
sc = ax.scatter(y / 1e3, z / 1e3, c=t_days, cmap="viridis", s=1.0)
ax.set_xlabel("y / 10³ km (in-plane lateral, perpendicular to Sun-Earth line)")
ax.set_ylabel("z / 10³ km (out-of-ecliptic)")
ax.set_title("ACE Modified Halo Orbit about L1 (Lissajous, 5-year span)")
ax.set_aspect("equal")
cb = plt.colorbar(sc, ax=ax, pad=0.02)
cb.set_label("Days since L+115")
ax.axhline(0, color="k", lw=0.5)
ax.axvline(0, color="k", lw=0.5)
plt.tight_layout()
plt.show()

print(f"Major axis (y full): {2 * y.max() / 1e3:.0f} × 10³ km")
print(f"Minor axis (z full): {2 * z.max() / 1e3:.0f} × 10³ km")

Because the planar and vertical libration periods are slightly different, the trajectory does not close — it densely fills a Lissajous figure within the ~150,000 × 75,000 km box, exactly as Figure 4 of the paper depicts.

면내·수직 진동 주기가 약간 다르기 때문에 궤적은 닫히지 않고 ~150,000 × 75,000 km 박스 안에서 Lissajous 도형을 조밀하게 채운다 — 논문의 Figure 4 그대로.

---
## 2. Time-of-Flight Composition Analysis (ULEIS-style) / TOF 조성 분석 (ULEIS 스타일)

ULEIS measures ion mass M from the time-of-flight $\tau$ over a known path $L$ and the residual energy $E$:

$$M = \frac{2 E}{v^2} = \frac{2 E \tau^2}{L^2}$$

ULEIS의 비행 경로는 $L$ = 50 cm. 비행 시간 $\tau$와 잔여 에너지 $E$로부터 이온 질량 $M$ 결정.

We simulate a population of incident ions (mixed H, He, C, O, Ne with two Ne isotopes) and plot them on the TOF–E plane. Curves of constant mass M form hyperbolic loci $\tau = L \sqrt{M / (2E)}$.

혼합된 H, He, C, O, Ne (²⁰Ne, ²²Ne) 이온 군집을 시뮬레이션하여 TOF-E 평면에 그리고, 등질량 곡선 $\tau = L \sqrt{M/(2E)}$를 표시한다.

In [ ]:
# Physical constants in convenient units.
AMU_KG = 1.66053906660e-27        # kg
MEV_TO_J = 1.602176634e-13         # J / MeV
FLIGHT_PATH_M = 0.50               # ULEIS flight path length, 50 cm


def tof_from_energy_mass(
    energy_mev: np.ndarray,
    mass_amu: float,
    flight_path_m: float = FLIGHT_PATH_M,
) -> np.ndarray:
    """Compute non-relativistic time-of-flight in nanoseconds.

    Args:
        energy_mev: Total kinetic energy in MeV.
        mass_amu: Ion mass in atomic mass units.
        flight_path_m: Flight path length in meters.

    Returns:
        Time of flight in nanoseconds.
    """
    e_j = energy_mev * MEV_TO_J
    m_kg = mass_amu * AMU_KG
    v = np.sqrt(2.0 * e_j / m_kg)
    return flight_path_m / v * 1e9


def simulate_uleis_population(
    n_per_species: int = 800,
    energy_min_mev: float = 0.3,
    energy_max_mev: float = 30.0,
    timing_jitter_ns: float = 0.5,
    energy_relative_jitter: float = 0.03,
    rng: np.random.Generator | None = None,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Generate a synthetic ULEIS population.

    Five species (H, He, C, O, Ne-20, Ne-22) are sampled from log-uniform
    energy distributions, then perturbed by Gaussian timing and energy jitter
    to mimic detector resolution.

    Args:
        n_per_species: Number of ions per species.
        energy_min_mev: Lower bound of energy distribution (MeV).
        energy_max_mev: Upper bound of energy distribution (MeV).
        timing_jitter_ns: 1-sigma timing resolution in nanoseconds.
        energy_relative_jitter: 1-sigma fractional energy resolution.
        rng: NumPy random generator for reproducibility.

    Returns:
        Tuple (energy_meas_mev, tof_meas_ns, species_label) of length
        (5 * n_per_species).
    """
    if rng is None:
        rng = np.random.default_rng(seed=42)
    species = [
        ("H", 1.008),
        ("He", 4.003),
        ("C-12", 12.000),
        ("O-16", 15.995),
        ("Ne-20", 19.992),
        ("Ne-22", 21.991),
    ]
    energies, tofs, labels = [], [], []
    log_e_min = np.log10(energy_min_mev)
    log_e_max = np.log10(energy_max_mev)
    for name, mass_amu in species:
        e_true = 10.0 ** rng.uniform(log_e_min, log_e_max, size=n_per_species)
        tof_true = tof_from_energy_mass(e_true, mass_amu)
        e_meas = e_true * (1.0 + rng.normal(0.0, energy_relative_jitter, size=n_per_species))
        tof_meas = tof_true + rng.normal(0.0, timing_jitter_ns, size=n_per_species)
        energies.append(e_meas)
        tofs.append(tof_meas)
        labels.append(np.full(n_per_species, name))
    return (
        np.concatenate(energies),
        np.concatenate(tofs),
        np.concatenate(labels),
    )


energy_mev, tof_ns, species = simulate_uleis_population()
print(f"Total simulated ions: {len(energy_mev)}")
print(f"Energy range: {energy_mev.min():.2f} to {energy_mev.max():.2f} MeV")
print(f"TOF    range: {tof_ns.min():.1f} to {tof_ns.max():.1f} ns")

In [ ]:
# Plot TOF vs energy with reference mass curves.
fig, ax = plt.subplots(figsize=(9, 6.5))
colors = {
    "H": "#1f77b4",
    "He": "#2ca02c",
    "C-12": "#9467bd",
    "O-16": "#ff7f0e",
    "Ne-20": "#d62728",
    "Ne-22": "#8c564b",
}
for name, color in colors.items():
    mask = species == name
    ax.scatter(energy_mev[mask], tof_ns[mask], s=2, alpha=0.4, color=color, label=name)

# Overplot constant-mass tracks.
e_grid = np.logspace(np.log10(0.3), np.log10(40.0), 200)
for label_mass, color, mass_amu in [
    ("M=1",  "#1f77b4", 1.008),
    ("M=4",  "#2ca02c", 4.003),
    ("M=12", "#9467bd", 12.0),
    ("M=16", "#ff7f0e", 16.0),
    ("M=20", "#d62728", 20.0),
    ("M=22", "#8c564b", 22.0),
]:
    ax.plot(e_grid, tof_from_energy_mass(e_grid, mass_amu),
            color=color, lw=1.0, ls="--")

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("Total Kinetic Energy E (MeV)")
ax.set_ylabel("Time of Flight τ (ns)")
ax.set_title("Synthetic ULEIS TOF–E plane (50 cm path)")
ax.legend(loc="upper right", framealpha=0.9, ncol=2)
plt.tight_layout()
plt.show()

Each species lies on its own hyperbolic track $\tau \propto \sqrt{M/E}$. Heavier ions have longer TOF at fixed energy.

각 species는 자신의 hyperbolic track $\tau \propto \sqrt{M/E}$ 위에 위치. 같은 에너지에서 무거운 이온일수록 TOF가 더 길다.

### 2.1 Reconstructed-mass histogram / 재구성 질량 히스토그램

Inverting $M = 2 E \tau^2 / L^2$ for each event recovers the ion mass to within the detector resolution. We focus on the Ne region to demonstrate $^{20}\!Ne$ vs $^{22}\!Ne$ separation — a primary ULEIS science goal.

각 이벤트에 대해 $M = 2E\tau^2/L^2$로 inverting하면 검출기 분해능 이내로 이온 질량 복원. Ne 영역에 집중하여 $^{20}\!Ne$ vs $^{22}\!Ne$ 분리 시연 — ULEIS의 주요 과학 목표.

In [ ]:
def reconstruct_mass_amu(
    energy_mev: np.ndarray,
    tof_ns: np.ndarray,
    flight_path_m: float = FLIGHT_PATH_M,
) -> np.ndarray:
    """Reconstruct ion mass in amu from TOF and energy.

    Args:
        energy_mev: Measured kinetic energy in MeV.
        tof_ns: Measured time-of-flight in nanoseconds.
        flight_path_m: Flight path length in meters.

    Returns:
        Reconstructed mass in atomic mass units.
    """
    tof_s = tof_ns * 1e-9
    v = flight_path_m / tof_s
    e_j = energy_mev * MEV_TO_J
    m_kg = 2.0 * e_j / v ** 2
    return m_kg / AMU_KG


mass_recon = reconstruct_mass_amu(energy_mev, tof_ns)
ne_mask = np.isin(species, ["Ne-20", "Ne-22"])

fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))

# All-species histogram.
axes[0].hist(mass_recon, bins=120, range=(0.5, 25.0), color="steelblue", alpha=0.85)
axes[0].set_xlabel("Reconstructed mass (amu)")
axes[0].set_ylabel("Counts")
axes[0].set_title("All species — reconstructed mass")
for m_true in [1, 4, 12, 16, 20, 22]:
    axes[0].axvline(m_true, color="red", lw=0.6, ls=":")

# Ne-only zoom.
axes[1].hist(mass_recon[ne_mask], bins=80, range=(18.0, 24.0), color="firebrick", alpha=0.85)
axes[1].set_xlabel("Reconstructed mass (amu)")
axes[1].set_ylabel("Counts")
axes[1].set_title("Ne isotope separation: ²⁰Ne vs ²²Ne")
axes[1].axvline(19.992, color="black", lw=0.8, ls=":", label="²⁰Ne")
axes[1].axvline(21.991, color="black", lw=0.8, ls="--", label="²²Ne")
axes[1].legend()

plt.tight_layout()
plt.show()

# Resolution check.
ne20 = mass_recon[species == "Ne-20"]
ne22 = mass_recon[species == "Ne-22"]
print(f"²⁰Ne reconstructed mean = {ne20.mean():.3f} amu, std = {ne20.std():.3f}")
print(f"²²Ne reconstructed mean = {ne22.mean():.3f} amu, std = {ne22.std():.3f}")
print(f"Mass separation Δ = {ne22.mean() - ne20.mean():.3f} amu (truth ≈ 2.0)")

ULEIS's σ_M ≈ 0.3 amu specification cleanly resolves $^{20}\!Ne$ from $^{22}\!Ne$ — exactly the kind of measurement that anchored ACE's solar-wind isotope studies.

ULEIS의 σ_M ≈ 0.3 amu 사양은 $^{20}\!Ne$과 $^{22}\!Ne$을 깔끔히 분해 — ACE의 태양풍 동위원소 연구를 anchored한 측정 유형.

---
## 3. dE/dx vs E Composition Analysis (CRIS/SIS-style) / dE/dx vs E 분석

For non-relativistic ions stopping in a stack of detectors, the Bethe-Bloch energy-loss rate scales as

$$\frac{dE}{dx} \propto \frac{Z^2}{v^2} \propto \frac{Z^2 M}{E}$$

Plotting (dE/dx, residual E) for many incident ions yields one curve per species (Z, M). This is the principle behind CRIS, SIS, EPAM, and SEPICA's dE/dx-E telescope arms.

비상대론적 이온이 검출기 스택에서 정지할 때 Bethe-Bloch 에너지 손실률은 $dE/dx \propto Z^2/v^2 \propto Z^2 M / E$. 다수 이온의 (dE/dx, residual E) 산점도는 species (Z, M)별로 곡선을 형성. 이것이 CRIS, SIS, EPAM, SEPICA의 dE/dx-E 텔레스코프 원리.

In [ ]:
def dedx_proxy(
    energy_mev_per_nuc: np.ndarray,
    z: int,
    mass_amu: float,
) -> np.ndarray:
    """Compute simplified Bethe-Bloch dE/dx proxy in MeV/(mg/cm²).

    Uses dE/dx ∝ Z² / v² ∝ Z² / (E/M) and includes a mild logarithmic
    correction to mimic the leading log term. The absolute normalization is
    arbitrary — the diagnostic value is the relative spacing of curves for
    different (Z, M).

    Args:
        energy_mev_per_nuc: Kinetic energy per nucleon (MeV/nuc).
        z: Nuclear charge of incident ion.
        mass_amu: Ion mass in atomic mass units.

    Returns:
        dE/dx proxy in MeV per mg/cm² (arbitrary normalization).
    """
    # Avoid singularities at low energy.
    e_safe = np.maximum(energy_mev_per_nuc, 1e-3)
    log_factor = np.log(1.0 + 4.0 * e_safe)
    return 0.30 * z ** 2 / e_safe * log_factor * mass_amu / 12.0


def simulate_dedx_population(
    species_list: list[tuple[str, int, float]],
    n_per_species: int = 600,
    e_min_mev_nuc: float = 1.0,
    e_max_mev_nuc: float = 200.0,
    rel_jitter: float = 0.05,
    rng: np.random.Generator | None = None,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Simulate a CRIS/SIS-style population in the dE/dx vs E plane.

    Args:
        species_list: List of (name, Z, mass_amu) tuples.
        n_per_species: Number of ions per species.
        e_min_mev_nuc: Lower energy bound (MeV/nucleon).
        e_max_mev_nuc: Upper energy bound (MeV/nucleon).
        rel_jitter: Relative Gaussian jitter on dE/dx and E.
        rng: NumPy random generator.

    Returns:
        Tuple (energy_total_mev, dedx_proxy, species_name).
    """
    if rng is None:
        rng = np.random.default_rng(seed=2024)
    energies, dedx_vals, names = [], [], []
    log_min = np.log10(e_min_mev_nuc)
    log_max = np.log10(e_max_mev_nuc)
    for name, z, mass_amu in species_list:
        e_per_nuc = 10.0 ** rng.uniform(log_min, log_max, size=n_per_species)
        dedx = dedx_proxy(e_per_nuc, z, mass_amu)
        # Convert to total energy and apply detector jitter.
        e_total = e_per_nuc * mass_amu * (1.0 + rng.normal(0.0, rel_jitter, size=n_per_species))
        dedx_meas = dedx * (1.0 + rng.normal(0.0, rel_jitter, size=n_per_species))
        energies.append(e_total)
        dedx_vals.append(dedx_meas)
        names.append(np.full(n_per_species, name))
    return (
        np.concatenate(energies),
        np.concatenate(dedx_vals),
        np.concatenate(names),
    )


# Selected species spanning H to Fe with mainline isotopes.
SPECIES = [
    ("H",   1, 1.008),
    ("He",  2, 4.003),
    ("C",   6, 12.000),
    ("O",   8, 16.000),
    ("Ne", 10, 20.180),
    ("Mg", 12, 24.305),
    ("Si", 14, 28.086),
    ("Fe", 26, 55.845),
]
energy_total, dedx_meas, names = simulate_dedx_population(SPECIES)
print(f"Simulated {len(energy_total)} ions across {len(SPECIES)} species.")

In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 6.5))
palette = plt.cm.viridis(np.linspace(0.05, 0.95, len(SPECIES)))
for (name, z, mass_amu), color in zip(SPECIES, palette):
    mask = names == name
    ax.scatter(energy_total[mask], dedx_meas[mask], s=2, alpha=0.55,
               color=color, label=f"{name} (Z={z})")
    # Reference track.
    e_ref_per_nuc = np.logspace(0, 2.4, 200)
    e_total_ref = e_ref_per_nuc * mass_amu
    dedx_ref = dedx_proxy(e_ref_per_nuc, z, mass_amu)
    ax.plot(e_total_ref, dedx_ref, color=color, lw=0.8, ls="--")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("Total Kinetic Energy E (MeV)")
ax.set_ylabel("dE/dx (arbitrary units)")
ax.set_title("Synthetic CRIS/SIS-style dE/dx vs E plane")
ax.legend(loc="lower left", ncol=2, framealpha=0.9, fontsize=9)
plt.tight_layout()
plt.show()

Each species occupies a distinct hyperbolic track in the (E, dE/dx) plane, sorted vertically by Z² and shifted horizontally by mass M. CRIS and SIS use exactly this technique — coupled with stacked Si detectors and high mass resolution — to identify isotopes across $2 \le Z \le 30$.

각 species는 (E, dE/dx) 평면에서 distinct hyperbolic track 점유. Z²로 수직 정렬, 질량 M로 수평 이동. CRIS와 SIS는 정확히 이 기법을 사용 — 적층 Si 검출기와 높은 질량 분해능과 결합 — $2 \le Z \le 30$ 동위원소 식별.

---
## 4. ACE Energy/Z Coverage Map / ACE 에너지/Z 커버리지 지도

Reproduce a simplified version of Figure 5 (Stone et al. 1998) showing which ACE instrument covers which (Z, energy) region for elemental composition.

Stone et al. (1998) Figure 5의 간소화 버전을 재현하여 어느 ACE 기기가 (Z, 에너지) 영역에서 원소 조성을 담당하는지 시각화.

In [ ]:
# (instrument, Z_min, Z_max, log10(E_min) MeV/nuc, log10(E_max) MeV/nuc, color)
instrument_boxes = [
    ("SWIMS",   2, 30, -4, -1, "#7f0000"),
    ("SWICS",   2, 30, -4, -1, "#ff7f0e"),
    ("ULEIS",   2, 28, -1,  1, "#ffd11a"),
    ("SEPICA",  2, 28, -1,  1, "#2ca02c"),
    ("SIS",     2, 30,  1,  2, "#17becf"),
    ("CRIS",    2, 30,  2,  3, "#1f77b4"),
]

fig, ax = plt.subplots(figsize=(10, 6.0))
for name, z_min, z_max, log_e_min, log_e_max, color in instrument_boxes:
    width = log_e_max - log_e_min
    height = z_max - z_min
    rect = plt.Rectangle((log_e_min, z_min), width, height,
                          facecolor=color, alpha=0.5, edgecolor="black", lw=0.7)
    ax.add_patch(rect)
    ax.text(log_e_min + width * 0.5, z_min + height * 0.5, name,
            ha="center", va="center", fontsize=10, fontweight="bold")

ax.set_xlim(-4.5, 3.5)
ax.set_ylim(0, 32)
ax.set_xlabel("log₁₀ Kinetic Energy (MeV/nucleon)")
ax.set_ylabel("Atomic Number Z")
ax.set_title("ACE Elemental-Composition Coverage (simplified Stone+1998 Fig. 5)")
ax.grid(True, alpha=0.3)
ax.axhline(28, color="red", ls=":", lw=0.8)
ax.text(-4.4, 28.3, "Z = 28 (Ni) baseline", color="red", fontsize=8)
plt.tight_layout()
plt.show()

print("Energy coverage spans roughly 9 decades (10^-4 to 10^3 MeV/nuc), as the paper claims.")

---
## 5. Summary / 요약

We have reproduced three core ACE-mission concepts:

1. **L1 geometry**: Solving the collinear restricted three-body balance equation gives $r_{L1} \approx 1.5 \times 10^6$ km ≈ 235 R_E, matching the paper's quoted ~240 R_E. Combined with $v_{SW}$ ≈ 400-800 km/s, ACE provides ~30-60 minutes of solar-wind lead time to NOAA.

2. **TOF composition (ULEIS)**: Synthetic events on the (E, τ) plane lie on hyperbolic mass tracks; reconstructing $M = 2 E \tau^2 / L^2$ separates $^{20}\!Ne$ from $^{22}\!Ne$ within ULEIS's quoted σ_M ≈ 0.3 amu resolution.

3. **dE/dx-E composition (CRIS, SIS, EPAM, SEPICA)**: The Bethe-Bloch scaling $dE/dx \propto Z^2/v^2$ produces species-specific tracks in the (E, dE/dx) plane, the foundation of stacked-Si telescopes.

Together these techniques anchor ACE's role as the most comprehensive composition observatory at L1 — and by extension the operational backbone of US space-weather monitoring through the RTSW broadcast.

ACE 미션의 세 가지 핵심 개념을 재현했다:
1. **L1 기하**: collinear restricted 3-body 균형 방정식 풀이로 $r_{L1} \approx 1.5 \times 10^6$ km ≈ 235 R_E. $v_{SW}$ ≈ 400-800 km/s와 결합하여 ACE는 NOAA에 ~30-60분 사전 경고 제공.
2. **TOF 조성 (ULEIS)**: (E, τ) 평면의 합성 이벤트는 hyperbolic mass track 위에 위치; $M = 2 E \tau^2 / L^2$ 재구성으로 ULEIS의 σ_M ≈ 0.3 amu 분해능 내에서 $^{20}\!Ne$과 $^{22}\!Ne$ 분리.
3. **dE/dx-E 조성 (CRIS, SIS, EPAM, SEPICA)**: Bethe-Bloch 스케일링 $dE/dx \propto Z^2/v^2$으로 species 특이 트랙 형성, 적층 Si 텔레스코프의 기초.

이 기법들이 함께 ACE를 L1에서 가장 종합적인 조성 관측소로 anchored하며, RTSW 송출을 통해 미국 우주 날씨 모니터링의 운영 백본 역할을 한다.